# 11 — Reformulação multiclasse (smoke)

**Fase 1** da reformulação metodológica: responder à recomendação 0 da orientadora (treinar OneVsRest e multiclasse 7+other).

**Escopo (estritamente):**
- TF-IDF + LogReg multiclasse, 2 estratégias (`native` softmax e `ovr`).
- Smoke em 1k amostras de treino — val inteiro preserva distribuição natural.
- Reaproveita os splits binários existentes (`artifacts/splits/*.parquet`) — só muda o `y`.

**Fora do escopo:**
- BERT multiclasse, random search, cross-validation, Qwen, análise de FPs sistematizada, comparação com binário, teste no `test.parquet`.

In [1]:
from pathlib import Path

import pandas as pd

from economy_classifier.datasets import (
    MULTICLASS_TOP7,
    OTHER_LABEL,
    attach_multiclass_label,
)
from economy_classifier.evaluation import (
    compute_confusion_matrix,
    compute_multiclass_metrics,
)
from economy_classifier.tfidf import (
    TfidfMulticlassConfig,
    train_tfidf_multiclass,
)

In [2]:
SMOKE_N = 1000
SEED = 42
REPO_ROOT = Path.cwd().parent
SPLITS_DIR = REPO_ROOT / "artifacts" / "splits"
RUN_DIR_NATIVE = REPO_ROOT / "artifacts" / "runs" / "11_multiclasse_smoke_native"
RUN_DIR_OVR = REPO_ROOT / "artifacts" / "runs" / "11_multiclasse_smoke_ovr"
LABELS = list(MULTICLASS_TOP7) + [OTHER_LABEL]
print("labels:", LABELS)

labels: ['poder', 'colunas', 'mercado', 'esporte', 'mundo', 'cotidiano', 'ilustrada', 'outros']


In [3]:
train = pd.read_parquet(SPLITS_DIR / "train.parquet")
val = pd.read_parquet(SPLITS_DIR / "val.parquet")

assert "category" in train.columns, (
    "category column missing — re-run 01_preparacao_dados.ipynb to persist it."
)

train = attach_multiclass_label(train)
val = attach_multiclass_label(val)

print("train shape:", train.shape)
print("val shape:", val.shape)
print("\ntrain label_multi distribution:")
print(train["label_multi"].value_counts())
print("\nval label_multi distribution:")
print(val["label_multi"].value_counts())

train shape: (106424, 8)
val shape: (26606, 8)

train label_multi distribution:
label_multi
outros       20734
poder        14015
colunas      13812
mercado      13421
esporte      12537
mundo        10984
cotidiano    10957
ilustrada     9964
Name: count, dtype: int64[pyarrow]

val label_multi distribution:
label_multi
outros       5166
poder        3608
colunas      3439
mercado      3355
esporte      3175
mundo        2689
cotidiano    2661
ilustrada    2513
Name: count, dtype: int64[pyarrow]


In [4]:
# Stratified smoke sample on train. Val is kept whole to preserve natural
# distribution (CLAUDE.md guard-rail: no balancing on val/test).
per_class = max(1, SMOKE_N // len(LABELS))
smoke_parts = []
for label in LABELS:
    pool = train[train["label_multi"] == label]
    take = min(per_class, len(pool))
    smoke_parts.append(pool.sample(n=take, random_state=SEED))
train_smoke = pd.concat(smoke_parts).sample(frac=1, random_state=SEED)
print("smoke train shape:", train_smoke.shape)
print(train_smoke["label_multi"].value_counts())

smoke train shape: (1000, 8)
label_multi
mundo        125
cotidiano    125
esporte      125
ilustrada    125
colunas      125
poder        125
outros       125
mercado      125
Name: count, dtype: int64[pyarrow]


In [5]:
config_native = TfidfMulticlassConfig(strategy="native")
result_native = train_tfidf_multiclass(
    train_smoke, val,
    label_column="label_multi",
    run_dir=RUN_DIR_NATIVE,
    config=config_native,
)
print("metrics:", result_native["metrics"])
print("vocab size:", result_native["vocabulary_size"])
print("timing:", result_native["timing"])

metrics: {'accuracy': 0.734, 'macro_f1': 0.7373}
vocab size: 50000
timing: {'train_seconds': 2.09, 'inference_seconds': 15.3}


In [6]:
preds_native = result_native["predictions"]
metrics_native = compute_multiclass_metrics(
    preds_native["y_true"], preds_native["y_pred"], labels=LABELS,
)
cm_native = compute_confusion_matrix(
    preds_native["y_true"], preds_native["y_pred"],
    labels=LABELS, normalize="true",
)
print("NATIVE — macro_f1:", metrics_native["macro_f1"],
      "weighted_f1:", metrics_native["weighted_f1"],
      "accuracy:", metrics_native["accuracy"])
print("\nper-class F1:")
for k, v in metrics_native["per_class_f1"].items():
    print(f"  {k:12s} {v}")
print("\nconfusion matrix (rows=true, cols=pred, normalized by row):")
cm_native.round(3)

NATIVE — macro_f1: 0.7373 weighted_f1: 0.7272 accuracy: 0.734

per-class F1:
  poder        0.7811
  colunas      0.5454
  mercado      0.7368
  esporte      0.9003
  mundo        0.8225
  cotidiano    0.7401
  ilustrada    0.734
  outros       0.6384

confusion matrix (rows=true, cols=pred, normalized by row):


,poder,colunas,mercado,esporte,mundo,cotidiano,ilustrada,outros
poder,0.836,0.028,0.034,0.004,0.012,0.057,0.006,0.022
colunas,0.129,0.479,0.115,0.056,0.042,0.035,0.074,0.070
mercado,0.069,0.035,0.763,0.001,0.021,0.033,0.019,0.058
esporte,0.018,0.011,0.009,0.906,0.011,0.024,0.016,0.005
mundo,0.012,0.028,0.029,0.001,0.861,0.028,0.024,0.017
cotidiano,0.041,0.023,0.026,0.006,0.009,0.810,0.030,0.055
ilustrada,0.011,0.037,0.013,0.004,0.018,0.028,0.859,0.031
outros,0.039,0.090,0.060,0.019,0.051,0.068,0.131,0.542


In [7]:
config_ovr = TfidfMulticlassConfig(strategy="ovr")
result_ovr = train_tfidf_multiclass(
    train_smoke, val,
    label_column="label_multi",
    run_dir=RUN_DIR_OVR,
    config=config_ovr,
)
print("metrics:", result_ovr["metrics"])
print("vocab size:", result_ovr["vocabulary_size"])
print("timing:", result_ovr["timing"])

metrics: {'accuracy': 0.7295, 'macro_f1': 0.7315}
vocab size: 50000
timing: {'train_seconds': 7.42, 'inference_seconds': 14.39}


In [8]:
preds_ovr = result_ovr["predictions"]
metrics_ovr = compute_multiclass_metrics(
    preds_ovr["y_true"], preds_ovr["y_pred"], labels=LABELS,
)
cm_ovr = compute_confusion_matrix(
    preds_ovr["y_true"], preds_ovr["y_pred"],
    labels=LABELS, normalize="true",
)
print("OVR — macro_f1:", metrics_ovr["macro_f1"],
      "weighted_f1:", metrics_ovr["weighted_f1"],
      "accuracy:", metrics_ovr["accuracy"])
print("\nper-class F1:")
for k, v in metrics_ovr["per_class_f1"].items():
    print(f"  {k:12s} {v}")
print("\nconfusion matrix (rows=true, cols=pred, normalized by row):")
cm_ovr.round(3)

OVR — macro_f1: 0.7315 weighted_f1: 0.721 accuracy: 0.7295

per-class F1:
  poder        0.7756
  colunas      0.5272
  mercado      0.7309
  esporte      0.9003
  mundo        0.8211
  cotidiano    0.7375
  ilustrada    0.7283
  outros       0.6312

confusion matrix (rows=true, cols=pred, normalized by row):


,poder,colunas,mercado,esporte,mundo,cotidiano,ilustrada,outros
poder,0.845,0.024,0.033,0.005,0.012,0.054,0.007,0.020
colunas,0.142,0.445,0.126,0.058,0.044,0.035,0.080,0.069
mercado,0.074,0.032,0.763,0.001,0.022,0.033,0.020,0.054
esporte,0.019,0.009,0.009,0.910,0.011,0.024,0.014,0.004
mundo,0.013,0.023,0.029,0.001,0.864,0.028,0.025,0.016
cotidiano,0.044,0.021,0.026,0.007,0.009,0.807,0.031,0.055
ilustrada,0.012,0.033,0.014,0.004,0.018,0.028,0.863,0.029
outros,0.044,0.081,0.063,0.020,0.053,0.071,0.138,0.530


## Conclusao da Fase 1 (smoke)

A pipeline multiclasse 7+other roda end-to-end: dados → splits existentes → mapeamento → treino → avaliacao → persistencia. Nada alem disso foi feito.

**O que esta entrega habilita (insumos para decidir Fase 2):**
- Macro-F1 das 8 classes em val, com 1k de treino balanceado.
- Matriz de confusao normalizada — base para a analise sistematizada de FPs (recomendacao 3 da orientadora).
- Comparacao native vs OvR para decidir a estrategia padrao na Fase 2 (BERT multiclasse).

**O que foi explicitamente deixado de fora:**
- BERT multiclasse, random search, cross-validation 90/10, Qwen, ensembles multiclasse, teste no `test.parquet`, atualizacao do `artigo_ieee.md`.
- Comparacao com binario — vira contraste artificial enquanto nao houver BERT e CV.

**Criterio de parada da Fase 1 atingido:** humano (orientadora) le este notebook e decide o escopo da Fase 2.

In [9]:
import numpy as np

def top_offdiag(cm: pd.DataFrame, k: int = 3) -> list[tuple[str, str, float]]:
    matrix = cm.to_numpy().copy()
    np.fill_diagonal(matrix, 0.0)
    flat = [
        (cm.index[i], cm.columns[j], matrix[i, j])
        for i in range(matrix.shape[0]) for j in range(matrix.shape[1])
    ]
    return sorted(flat, key=lambda x: x[2], reverse=True)[:k]


print("=== Resumo Fase 1 — TF-IDF + LogReg multiclasse (smoke 1k) ===\n")
print(f"{'estrategia':<12} {'macro_f1':<10} {'weighted_f1':<12} {'accuracy':<10}")
print(f"{'native':<12} {metrics_native['macro_f1']:<10} "
      f"{metrics_native['weighted_f1']:<12} {metrics_native['accuracy']:<10}")
print(f"{'ovr':<12} {metrics_ovr['macro_f1']:<10} "
      f"{metrics_ovr['weighted_f1']:<12} {metrics_ovr['accuracy']:<10}")

print("\n--- top-3 confusoes off-diagonal (NATIVE) ---")
for true, pred, score in top_offdiag(cm_native, k=3):
    print(f"  {true:10s} -> {pred:10s} : {score:.3f}")

print("\n--- top-3 confusoes off-diagonal (OVR) ---")
for true, pred, score in top_offdiag(cm_ovr, k=3):
    print(f"  {true:10s} -> {pred:10s} : {score:.3f}")

=== Resumo Fase 1 — TF-IDF + LogReg multiclasse (smoke 1k) ===

estrategia   macro_f1   weighted_f1  accuracy  
native       0.7373     0.7272       0.734     
ovr          0.7315     0.721        0.7295    

--- top-3 confusoes off-diagonal (NATIVE) ---
  outros     -> ilustrada  : 0.131
  colunas    -> poder      : 0.129
  colunas    -> mercado    : 0.115

--- top-3 confusoes off-diagonal (OVR) ---
  colunas    -> poder      : 0.142
  outros     -> ilustrada  : 0.138
  colunas    -> mercado    : 0.126
